[Numpy 2.x support](https://github.com/catboost/catboost/issues/2671) for CatBoost

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from catboost import CatBoostRegressor

from var import DATA_OUT

In [ ]:
MAV_WINDOW = 5
LAG_WINDOW = 180
max_window = max(MAV_WINDOW, LAG_WINDOW)
hrs_, mins_ = divmod(max_window, 60)

df = pd.read_parquet(Path(DATA_OUT, 'df.parquet'), engine='pyarrow').drop(
    columns=['perc_mild_scint', 'perc_strong_scint', 's4_max']
)

# Pre-filtering
df = df[
    (df.index.hour > 17) | (df.index.hour < 6) | (
        (df.index.hour == (17 - hrs_)) & (df.index.minute >= (60 - mins_))
    )
]

# MAVs
df[f"s4_mean_ema_{MAV_WINDOW}m"] = (
    df['s4_mean'].ewm(span=MAV_WINDOW).mean()
)

# Lags
df[f"h_tmk_lag_{LAG_WINDOW}m"] = (
    df['h_tmk'].shift(LAG_WINDOW)
)

# Filtering
df = df[(df.index.hour >= 18) | (df.index.hour < 6)]

# Target
df['s4_mean_lead'] = df['s4_mean'].shift(-1)

In [ ]:
# df.loc['2022-01-01':'2023-01-31',['s4_mean','h_tmk']].isna().sum() / df.loc['2022-01-01':'2023-01-31'].shape[0]

In [ ]:
TRAIN_START, TRAIN_STOP = '2022-01-01', '2023-01-31'
CALIB_START, CALIB_STOP = '2023-03-01', '2023-03-17'
TEST_START, TEST_STOP = '2023-03-18', '2023-03-31'

In [ ]:
X_cols = [
    'n_sat',
    's4_mean',
    'field_magnitude_avg',
    'wind_speed',
    'wind_density',
    'wind_pressure',
    'eletric_field',
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_ema_5m',
    'h_tmk_lag_180m',
]

y_col = 's4_mean_lead'

X_train, X_calib, X_test = (
    df.loc[TRAIN_START:TRAIN_STOP, X_cols].copy(),
    df.loc[CALIB_START:CALIB_STOP, X_cols].copy(),
    df.loc[TEST_START:TEST_STOP, X_cols].copy(),
)
y_train, y_calib, y_test = (
    df.loc[TRAIN_START:TRAIN_STOP, y_col].copy().fillna(0),
    df.loc[CALIB_START:CALIB_STOP, y_col].copy().fillna(0),
    df.loc[TEST_START:TEST_STOP, y_col].copy().fillna(0),
)

## CatBoost quantile regression [(*a manina*)](https://stephane-degeye.medium.com/probabilistic-forecasting-i-temperature-f96ded1e7247)

In [ ]:
quantile_mapping = {
    95: [0.025, 0.975],
    90: [0.05, 0.95],
    80: [0.10, 0.90],
}

In [ ]:
quantile_str = str(quantile_mapping[95]).replace('[','').replace(']','')

[params](https://catboost.ai/docs/en/references/training-parameters/)

In [ ]:
cb_multiquantile = CatBoostRegressor(
    loss_function=f'MultiQuantile:alpha={quantile_str}',
    thread_count=-1,
    bootstrap_type="Bernoulli",
    sampling_frequency='PerTree',
    iterations=3_000,
    od_type='Iter',
    od_wait=300,
    # use_best_model=True,
    max_depth=10,
    subsample=0.9,
    colsample_bylevel=0.9,
    min_data_in_leaf=50,
    verbose=1,
    random_seed=42,
)

cb_multiquantile.fit(X_train, y_train)

In [ ]:
y_pred_calib = cb_multiquantile.predict(X_calib)
y_pred_test = cb_multiquantile.predict(X_test)

In [ ]:
level = 95
q_low = quantile_mapping[level][0]
q_hig = quantile_mapping[level][1]

print(f'level : {level} interval : [{q_low} , {q_hig}]\n')

# Keep only predicted Quantiles for the calibration set related to the current level
quantile_regression_calibration_intervals = np.zeros([len(X_calib), 2])
quantile_regression_calibration_intervals[:, 0] = y_pred_calib[:, 0]
quantile_regression_calibration_intervals[:, 1] = y_pred_calib[:, 1]

# Compute Non-Conformity Measures on the Calibration set (How predicted Quantiles relate to the True value)
non_conformity_scores = np.max(
    [
        quantile_regression_calibration_intervals[:, 0] - y_calib,
        y_calib - quantile_regression_calibration_intervals[:, 1],
    ],
    axis=0,
)

# Sort Non-Conformity Measures
non_conformity_scores = np.sort(non_conformity_scores)[::-1]

# Compute the Quantile based on Non-Conformity Measures distribution with level as threshold
emperical_quantile = (level/100) * (1 + (1 / len(y_calib)))
correction_factor = np.quantile(non_conformity_scores, emperical_quantile, method="higher")

# Plot the non_conformity_scores distribution
plt.hist(non_conformity_scores, bins='auto', color='magenta')

# Add a vertical line for the Quantile
plt.axvline(correction_factor, color='black', linestyle='dashed', linewidth=1, label='quantile')

plt.legend()
plt.xlabel('Calibration Error')
plt.ylabel('Frequency')
plt.title('Histogram of Calibration Errors')

plt.show()

# Keep only predicted Quantiles for the test set related to the current level
quantile_regression_prediction_intervals = np.zeros([len(X_test), 2])
quantile_regression_prediction_intervals[:, 0] = y_pred_test[:, 0]
quantile_regression_prediction_intervals[:, 1] = y_pred_test[:, 1]

# Apply Correction factor on test set
correction_factor_test = np.ones([len(y_test), 2])
correction_factor_test[:, 0] *= correction_factor
correction_factor_test[:, 1] *= correction_factor

y_pred_test[:,0] = quantile_regression_prediction_intervals[:, 0] - correction_factor_test[:, 0]
y_pred_test[:,1] = quantile_regression_prediction_intervals[:, 1] + correction_factor_test[:, 1]

In [ ]:
print(f'level : {level} interval : [{q_low} , {q_hig}]\n')

hrs_start = 23
hrs_stop = 26

plt.figure(figsize=(20, 10))
plt.plot(y_test.iloc[hrs_start*60:hrs_stop*60].values, color='tab:blue')

plt.fill_between(
    x=np.arange(stop=(hrs_stop-hrs_start)*60),
    y1=y_pred_test[hrs_start*60:hrs_stop*60,0],
    y2=y_pred_test[hrs_start*60:hrs_stop*60,1],
    color='gray',
    alpha=0.2,
    label='Prediction Interval',
)

plt.show()

In [ ]:
df_eval = pd.DataFrame()

In [ ]:
df_eval['y_test'] = y_test
df_eval['y_pred_low'] = y_pred_test[:,0].clip(0,)
df_eval['y_pred_hig'] = y_pred_test[:,1]
df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

In [ ]:
df_eval['is_covered'].sum() / df_eval.shape[0]

In [ ]:
df_eval['y_test'].le(df_eval['y_pred_hig']).sum() / df_eval.shape[0]

## CatBoost + ACI

In [ ]:
from mapie.subsample import BlockBootstrap

from scintill_ai.conformal import aci_ts_regressor_predict
from scintill_ai import LW_S4_THRESHOLD

In [ ]:
cv = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
cb = CatBoostRegressor(
    loss_function='RMSE',
    thread_count=-1,
    bootstrap_type="Bernoulli",
    sampling_frequency='PerTree',
    iterations=3_000,
    od_type='Iter',
    od_wait=300,
    # use_best_model=True,
    max_depth=10,
    subsample=0.9,
    colsample_bylevel=0.9,
    min_data_in_leaf=50,
    verbose=1,
    random_seed=42,
)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=cb,
    cv=cv,
    train_data=(
        pd.concat([X_train, X_calib]),
        pd.concat([y_train, y_calib]),
    ),
    test_data=(X_test, y_test),
    update_calibration=True,
    gamma=0.04,
    forecast_horizon=2,
    alpha_list=[1 - 0.95],
)

In [ ]:
def rrmse(y_true, y_pred, digit=3):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mean_true = np.mean(y_true)
    return np.round(rmse / mean_true, digit)

def rmse(y_true, y_pred, digit=3):
    return np.round(np.sqrt(np.mean((y_pred - y_true) ** 2)), digit)

In [ ]:
print(
    f"RMSE: {rmse(y_true=y_test.values, y_pred=aci_res[0]['y_pred'])} | RRMSE: {rrmse(y_true=y_test.values, y_pred=aci_res[0]['y_pred'])}"
)

In [ ]:
aci_res_loaded = aci_res[0]

In [ ]:
df_eval = pd.DataFrame({
    'y_pred': aci_res_loaded['y_pred'],
    'y_pi_low': aci_res_loaded['y_pis'][:, 0, 0],
    'y_pi_upp': aci_res_loaded['y_pis'][:, 1, 0],
    'y_true': y_test,
})

df_eval['is_scint'] = df_eval['y_true'].ge(LW_S4_THRESHOLD)
df_eval['is_scint_covered'] = df_eval['is_scint'] & (
    df_eval['y_pi_low'].le(df_eval['y_true'])) & (
    df_eval['y_true'].le(df_eval['y_pi_upp'])
)
df_eval['is_scint_covered_upward'] = df_eval['is_scint'] & (
    df_eval['y_true'].le(df_eval['y_pi_upp'])
)

In [ ]:
perc_scint = df_eval['is_scint'].eq(True).sum() / df_eval.shape[0]
perc_scint_covered = df_eval['is_scint_covered'].eq(True).sum() / df_eval['is_scint'].eq(True).sum()
perc_scint_covered_upward = df_eval['is_scint_covered_upward'].eq(True).sum() / df_eval['is_scint'].eq(True).sum()

In [ ]:
print(
    f'Scintillation happens {perc_scint:.1%} of the time. Scintillation is covered {perc_scint_covered:.1%} of the time (upward: {perc_scint_covered_upward:.1%})'
)

In [ ]:
plot_dict = aci_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    ls=':',
    c="tab:blue",
    label="Forecast",
)

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('CatBoost + ACI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
# ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_xlim(y_test.loc['2023-03-19 23'].index[0], y_test.loc['2023-03-20 00'].index[-1])
# ax.set_ylim(0, 0.85)

# plt.savefig('aci_.png', dpi=800, bbox_inches='tight')
plt.show()

## CatBoost + [CQR](https://github.com/superlinear-ai/conformal-tights?tab=readme-ov-file#predicting-quantiles) (conformal tights, vedi anche [ChatGPT](https://chatgpt.com/share/67d0257d-e400-8002-ab0e-cbed67551ea4))

In [ ]:
cb = CatBoostRegressor(
    loss_function='RMSE',
    thread_count=-1,
    bootstrap_type="Bernoulli",
    sampling_frequency='PerTree',
    iterations=3_000,
    od_type='Iter',
    od_wait=300,
    max_depth=10,
    subsample=0.9,
    colsample_bylevel=0.9,
    min_data_in_leaf=50,
    verbose=1,
    random_seed=42,
)

## CatBoost + AgACI